# BÀI TẬP THỰC HÀNH (Mục 4.1.4)
Chủ đề: Xác định tập phổ biến với giải thuật Apriori sử dụng thư viện `apyori`. Dữ liệu: `Market_Basket_Optimisation.csv` (Dữ liệu giỏ hàng siêu thị).

## 1. Cài đặt và Import thư viện

In [1]:
# Cài đặt thư viện apyori
# !pip install apyori

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from apyori import apriori

## 2. Nạp và Tiền xử lý dữ liệu
Khác với thư viện `mlxtend` yêu cầu dữ liệu dạng One-Hot Encoding (0/1), thư viện `apyori` yêu cầu dữ liệu đầu vào là một List of Lists (Danh sách các danh sách giao dịch), trong đó mỗi giao dịch là một danh sách các chuỗi (tên sản phẩm).

Dữ liệu `Market_Basket_Optimisation.csv` không có tiêu đề cột (header), dòng đầu tiên đã là dữ liệu mua hàng.

In [2]:
# Đọc dữ liệu, lưu ý header=None vì file không có dòng tiêu đề
dataset = pd.read_csv('Market_Basket_Optimisation.csv', header=None)

# Hiển thị kích thước dữ liệu
print("Kích thước dữ liệu:", dataset.shape)
print("\n5 giao dịch đầu tiên:")
print(dataset.head())

# --- CHUYỂN ĐỔI DỮ LIỆU SANG DẠNG LIST OF LISTS ---
# Tạo một danh sách chứa các giao dịch
transactions = []

# Duyệt qua tất cả các dòng (7501 dòng)
for i in range(0, 7501):
    # Lấy các sản phẩm trong dòng thứ i, chuyển thành chuỗi
    # Chỉ lấy các giá trị không phải là NaN (vì các dòng có độ dài khác nhau)
    row = [str(dataset.values[i, j]) for j in range(0, 20) if str(dataset.values[i, j]) != 'nan']
    transactions.append(row)

# Kiểm tra giao dịch đầu tiên sau khi chuyển đổi
print("\nGiao dịch đầu tiên (sau khi xử lý):")
print(transactions[0])

Kích thước dữ liệu: (7501, 20)

5 giao dịch đầu tiên:
              0          1           2                 3             4   \
0         shrimp    almonds     avocado    vegetables mix  green grapes   
1        burgers  meatballs        eggs               NaN           NaN   
2        chutney        NaN         NaN               NaN           NaN   
3         turkey    avocado         NaN               NaN           NaN   
4  mineral water       milk  energy bar  whole wheat rice     green tea   

                 5     6               7             8             9   \
0  whole weat flour  yams  cottage cheese  energy drink  tomato juice   
1               NaN   NaN             NaN           NaN           NaN   
2               NaN   NaN             NaN           NaN           NaN   
3               NaN   NaN             NaN           NaN           NaN   
4               NaN   NaN             NaN           NaN           NaN   

               10         11     12     13             1

## 3. Thực hiện giải thuật Apriori (Tìm tập phổ biến)
Chúng ta sẽ gọi hàm `apriori` từ thư viện `apyori`. Để tìm tập phổ biến, chúng ta cần xác định ngưỡng `min_support`.

- Thiết lập tham số:

    - min_support: Chọn 0.003 (Tương đương sản phẩm xuất hiện khoảng 3 lần/ngày * 7 ngày / 7500 giao dịch).

    - min_confidence: 0.2 (Độ tin cậy tối thiểu).

    - min_lift: 3 (Để tìm các mối quan hệ có ý nghĩa, lift > 1).

    - min_length: 2 (Chỉ quan tâm đến nhóm có từ 2 sản phẩm trở lên).

In [3]:
# Chạy thuật toán Apriori
rules = apriori(transactions, 
                min_support=0.003, 
                min_confidence=0.2, 
                min_lift=3, 
                min_length=2)

# Kết quả trả về là một generator, cần chuyển sang list để xem
results = list(rules)

print(f"Đã tìm thấy {len(results)} tập phổ biến và luật kết hợp.")

Đã tìm thấy 80 tập phổ biến và luật kết hợp.


## 4. Hiển thị kết quả Tập phổ biến
Kết quả trả về của `apyori` có cấu trúc khá phức tạp. Chúng ta cần trích xuất thông tin Items (Sản phẩm) và Support (Độ hỗ trợ) để hiển thị Tập phổ biến.

In [4]:
# Hàm để chuyển đổi kết quả raw từ apyori sang DataFrame của Pandas cho dễ đọc
def inspect_frequent_itemsets(results):
    lhs         = [] # Tập sản phẩm (Itemset)
    supports    = [] # Độ hỗ trợ (Support)

    for result in results:
        # result.items chứa frozenset các sản phẩm trong tập phổ biến
        itemset = result.items 
        support = result.support
        
        lhs.append(tuple(itemset))
        supports.append(support)
        
    return list(zip(lhs, supports))

# Tạo DataFrame
result_df = pd.DataFrame(inspect_frequent_itemsets(results), columns=['Itemset', 'Support'])

# Sắp xếp theo độ hỗ trợ giảm dần (những món hay mua cùng nhau nhất lên đầu)
result_df = result_df.sort_values(by='Support', ascending=False)

print("\nDANH SÁCH TẬP PHỔ BIẾN (TOP 10)")
print(result_df.head(10))


DANH SÁCH TẬP PHỔ BIẾN (TOP 10)
                                        Itemset   Support
4                  (herb & pepper, ground beef)  0.015998
26  (spaghetti, frozen vegetables, ground beef)  0.008666
7                (whole wheat pasta, olive oil)  0.007999
30   (shrimp, frozen vegetables, mineral water)  0.007199
48                 (spaghetti, olive oil, milk)  0.007199
38  (herb & pepper, mineral water, ground beef)  0.006666
34     (spaghetti, frozen vegetables, tomatoes)  0.006666
39      (herb & pepper, spaghetti, ground beef)  0.006399
32       (spaghetti, shrimp, frozen vegetables)  0.005999
43             (spaghetti, shrimp, ground beef)  0.005999
